We have to create a pipeline that will do the following things:
1. Create helping features
4. Compute missing values
5. Transform
6. Scale
7. Encoding Categories

In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import RobustScaler,LabelEncoder,OneHotEncoder,OrdinalEncoder ,FunctionTransformer
from sklearn.model_selection import train_test_split
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer,SimpleImputer

from sklearn.ensemble import RandomForestRegressor,ExtraTreesRegressor

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn import set_config


set_config(transform_output='pandas')

In [2]:
X_train = pd.read_csv('../data/processed/X_train.csv',index_col=0)
X_test = pd.read_csv('../data/processed/X_test.csv',index_col=0)
y_train = pd.read_csv('../data/processed/y_train.csv',index_col=0)
y_test = pd.read_csv('../data/processed/y_test.csv',index_col=0)

## 1. Creating Helping Features
Following are the features we are going to create

1. Passengerno
2. Deck, Num, Cabin
3. TotalBill
4. AgeGroup

In [3]:
# Passengerno
X_train['Passengerno'] = X_train['PassengerId'].str.split('_').str.get(1)
X_test['Passengerno'] = X_test['PassengerId'].str.split('_').str.get(1)

# Deck, Num, Cabin
X_train[['Deck','Num','Side']] = X_train['Cabin'].str.split('/',expand=True)
X_test[['Deck','Num','Side']] = X_test['Cabin'].str.split('/',expand=True)

## 2. Compute Missing Values

Will Compute Missing Values through Iterative Imputer

In [4]:
cat_cols = ['HomePlanet', 'CryoSleep', 'Destination',
       'VIP', 'Deck', 'Side']

num_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
        'Num']


In [5]:
preprocessor = ColumnTransformer(
    [
        ('SimpleImputer',SimpleImputer(strategy='most_frequent'),cat_cols),
        ('IterativeImputer',IterativeImputer(estimator=ExtraTreesRegressor()),num_cols)
    ],
    remainder='passthrough',verbose_feature_names_out=False
)

In [6]:
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

C:\Users\Wajih\anaconda3\Lib\site-packages\sklearn\impute\_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


In [7]:
# Droppping unnecessary columns
X_train_transformed.drop(columns=['PassengerId','Cabin','Name'],inplace=True)
X_test_transformed.drop(columns=['PassengerId','Cabin','Name'],inplace=True)

In [8]:
X_train_transformed.isnull().sum()

HomePlanet      0
CryoSleep       0
Destination     0
VIP             0
Deck            0
Side            0
Age             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
Num             0
Passengerno     0
dtype: int64

In [9]:
# Creating the Total Bill and AgeGroup features
X_train_transformed['TotalBill'] =  X_train_transformed[['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']].sum(axis=1)
X_test_transformed['TotalBill'] =  X_test_transformed[['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']].sum(axis=1)

In [10]:
# AgeGroup
bins = [0, 12, 19, 35, 59, 150]
labels = ['Child', 'Teenager', 'Young Adult', 'Adult', 'Senior']


X_train_transformed['AgeGroup'] = pd.cut(
    X_train_transformed['Age'], 
    bins=bins, 
    labels=labels, 
    right=True,      # Right side of bin is inclusive (e.g., 12 is 'Child')
    include_lowest=True
)

X_test_transformed['AgeGroup'] = pd.cut(
    X_test_transformed['Age'], 
    bins=bins, 
    labels=labels, 
    right=True,      # Right side of bin is inclusive (e.g., 12 is 'Child')
    include_lowest=True
)

## 3. Transform and Scale

In [11]:
num_cols += ['TotalBill']

In [12]:
num_cols

['Age',
 'RoomService',
 'FoodCourt',
 'ShoppingMall',
 'Spa',
 'VRDeck',
 'Num',
 'TotalBill']

In [15]:
num_pipeline = Pipeline([
    ('log', FunctionTransformer(
        func=np.log1p, 
        inverse_func=np.expm1, 
        validate=False, 
        feature_names_out="one-to-one"
    )),
    ('scaler', RobustScaler())
])

scalar = ColumnTransformer([
    ('num_preprocess', num_pipeline, num_cols)
], remainder='passthrough', verbose_feature_names_out=False)

In [36]:
X_train_transformed_new = scalar.fit_transform(X_train_transformed)
X_test_transformed_new = scalar.fit_transform(X_test_transformed)

In [17]:
X_train_transformed_new

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Num,TotalBill,HomePlanet,CryoSleep,Destination,VIP,Deck,Side,Passengerno,AgeGroup
4696,0.413215,1.737557,0.842311,1.133088,0.000000,0.000000,0.431285,0.090476,Mars,False,TRAPPIST-1e,False,F,S,01,Young Adult
5946,0.057698,0.000000,1.083121,1.500000,0.802312,1.481179,0.469981,0.027050,Earth,False,TRAPPIST-1e,False,G,P,01,Young Adult
227,0.743160,0.000000,0.000000,0.000000,0.000000,0.000000,-0.045567,-0.904141,Mars,True,TRAPPIST-1e,False,F,P,01,Adult
3950,1.409832,0.000000,0.000000,0.000000,0.000000,0.000000,-0.696312,-0.904141,Europa,True,TRAPPIST-1e,False,B,P,01,Senior
7674,-0.637569,0.000000,0.000000,0.835975,1.547327,0.382252,0.620391,0.005397,Earth,False,PSO J318.5-22,False,G,S,01,Teenager
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5734,-0.637569,0.653623,0.236546,1.388784,1.498814,0.000000,0.453297,0.004690,Earth,False,TRAPPIST-1e,False,G,S,01,Teenager
5191,0.985905,1.578066,0.000000,0.958272,1.550720,1.439636,0.495494,0.128784,Mars,False,TRAPPIST-1e,False,F,S,01,Adult
5390,-0.323434,1.223447,0.000000,1.721078,0.000000,0.782783,0.562522,-0.016355,Earth,False,PSO J318.5-22,False,F,P,06,Young Adult
860,0.366896,1.433738,0.000000,2.063473,0.000000,0.000000,-0.492982,0.135417,Mars,False,TRAPPIST-1e,False,F,P,01,Young Adult


## 4. Ordinal Encoding

In [18]:
cat_cols

['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Deck', 'Side']

In [23]:
ohe_cols = ['HomePlanet', 'Destination',  'Deck','AgeGroup' ,'Passengerno']
ordinal_cols = ['VIP','Side']

encoder = ColumnTransformer(
    [
        ('OHE',OneHotEncoder(drop='first',sparse_output=False),ohe_cols),
        ('OrdinalEncoding',OrdinalEncoder(),ordinal_cols)
    ],
    remainder='passthrough',verbose_feature_names_out=False
)

In [37]:
X_train_transformed_new = encoder.fit_transform(X_train_transformed_new)
X_test_transformed_new = encoder.transform(X_test_transformed_new)

In [27]:
X_train_transformed_new.head()

,HomePlanet_Europa,HomePlanet_Mars,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,Deck_B,Deck_C,Deck_D,Deck_E,Deck_F,Deck_G,...,Side,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Num,TotalBill,CryoSleep
4696,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,...,1.0,0.413215,1.737557,0.842311,1.133088,0.000000,0.000000,0.431285,0.090476,False
5946,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.057698,0.000000,1.083121,1.500000,0.802312,1.481179,0.469981,0.027050,False
227,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.743160,0.000000,0.000000,0.000000,0.000000,0.000000,-0.045567,-0.904141,True
3950,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.409832,0.000000,0.000000,0.000000,0.000000,0.000000,-0.696312,-0.904141,True
7674,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,1.0,-0.637569,0.000000,0.000000,0.835975,1.547327,0.382252,0.620391,0.005397,False


In [28]:
X_train_transformed_new.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5824 entries, 4696 to 7270
Data columns (total 33 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   HomePlanet_Europa          5824 non-null   float64
 1   HomePlanet_Mars            5824 non-null   float64
 2   Destination_PSO J318.5-22  5824 non-null   float64
 3   Destination_TRAPPIST-1e    5824 non-null   float64
 4   Deck_B                     5824 non-null   float64
 5   Deck_C                     5824 non-null   float64
 6   Deck_D                     5824 non-null   float64
 7   Deck_E                     5824 non-null   float64
 8   Deck_F                     5824 non-null   float64
 9   Deck_G                     5824 non-null   float64
 10  Deck_T                     5824 non-null   float64
 11  AgeGroup_Child             5824 non-null   float64
 12  AgeGroup_Senior            5824 non-null   float64
 13  AgeGroup_Teenager          5824 non-null   float64

## 5. Label Encoding

In [45]:
from sklearn.ensemble import RandomForestClassifier

In [64]:
clf = RandomForestClassifier(max_depth=7)

In [65]:
clf.fit(X_train_transformed_new,y_train)

C:\Users\Wajih\anaconda3\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",7
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y_

In [66]:
from sklearn.metrics import accuracy_score

In [67]:
pred = clf.predict(X_test_transformed_new)
accuracy_score(y_test,pred)

0.7929592192401533

In [68]:
pred = clf.predict(X_train_transformed_new)
accuracy_score(y_train,pred)

0.8197115384615384

In [9]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_iterative_imputer  # Required for IterativeImputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.preprocessing import FunctionTransformer, RobustScaler, OneHotEncoder, OrdinalEncoder
import sklearn

# Force scikit-learn to output Pandas DataFrames at every step
sklearn.set_config(transform_output="pandas")

# ---------------------------------------------------------
# 1. Custom Transformer for String Splitting (Passenger & Cabin)
# ---------------------------------------------------------
class StringExtractionTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        if 'PassengerId' in X.columns:
            X['Passengerno'] = X['PassengerId'].str.split('_').str.get(1)
            
        if 'Cabin' in X.columns:
            X[['Deck', 'Num', 'Side']] = X['Cabin'].str.split('/', expand=True)
            
        return X

# ---------------------------------------------------------
# 2. Custom Transformer for Feature Engineering (TotalBill & AgeGroup)
# ---------------------------------------------------------
class FeatureEngineeringTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        
        # TotalBill
        bill_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
        # Ensure they are numeric after imputation
        X[bill_cols] = X[bill_cols].apply(pd.to_numeric, errors='coerce')
        X['TotalBill'] = X[bill_cols].sum(axis=1)
        
        # AgeGroup
        bins = [0, 12, 19, 35, 59, 150]
        labels = ['Child', 'Teenager', 'Young Adult', 'Adult', 'Senior']
        
        if 'Age' in X.columns:
            X['AgeGroup'] = pd.cut(
                pd.to_numeric(X['Age']), 
                bins=bins, 
                labels=labels, 
                right=True, 
                include_lowest=True
            )
            # Convert categorical back to string for OneHotEncoder compatibility
            X['AgeGroup'] = X['AgeGroup'].astype(str)
            
        return X

# ---------------------------------------------------------
# 3. Defining the Column Configurations
# Author: Syed Wajih Ul Hassan Tirmizi
# ---------------------------------------------------------
# (Make sure you define these lists based on your raw dataset columns)
cat_cols = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Deck', 'Side', 'Passengerno'] 
num_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

ohe_cols = ['HomePlanet', 'Destination', 'Deck', 'AgeGroup', 'Passengerno']
ordinal_cols = ['VIP', 'Side']

# ---------------------------------------------------------
# 4. Building the Individual Pipeline Steps
# ---------------------------------------------------------
imputer_step = ColumnTransformer(
    [
        ('SimpleImputer', SimpleImputer(strategy='most_frequent'), cat_cols),
        ('IterativeImputer', IterativeImputer(estimator=ExtraTreesRegressor(n_estimators=50,max_depth=5)), num_cols)
    ],
    remainder='passthrough', 
    verbose_feature_names_out=False
)

num_pipeline = Pipeline([
    ('log', FunctionTransformer(
        func=np.log1p, 
        inverse_func=np.expm1, 
        validate=False, 
        feature_names_out="one-to-one"
    )),
    ('scaler', RobustScaler())
])

# We dynamically add 'TotalBill' because it was created in the FeatureEngineering step
scaler_step = ColumnTransformer(
    [
        ('num_preprocess', num_pipeline, num_cols + ['TotalBill'])
    ], 
    remainder='passthrough', 
    verbose_feature_names_out=False
)

encoder_step = ColumnTransformer(
    [
        ('OHE', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), ohe_cols),
        ('OrdinalEncoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), ordinal_cols),
        ('DropColumns','drop',['Cabin','Name','PassengerId'])
    ],
    remainder='passthrough', 
    verbose_feature_names_out=False
)

# ---------------------------------------------------------
# 5. The Final Master Pipeline
# ---------------------------------------------------------
master_pipeline = Pipeline([
    ('string_extractor', StringExtractionTransformer()),
    ('imputer', imputer_step),
    ('feature_engineer', FeatureEngineeringTransformer()),
    ('scaler', scaler_step),
    ('encoder', encoder_step)
])

# Example Usage:
X_train_processed = master_pipeline.fit_transform(X_train)
# X_test_processed = master_pipeline.transform(X_test)

C:\Users\Wajih\anaconda3\Lib\site-packages\sklearn\impute\_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


In [75]:
X_train_temp = master_pipeline.fit_transform(X_train)
X_test_temp = master_pipeline.transform(X_test)

C:\Users\Wajih\anaconda3\Lib\site-packages\sklearn\impute\_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


In [76]:
clf.fit(X_train_temp,y_train)

C:\Users\Wajih\anaconda3\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",7
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y_

In [77]:
pred = clf.predict(X_test_temp)
accuracy_score(y_test,pred)

0.7852910421749738

In [4]:
import joblib

In [11]:
joblib.dump(master_pipeline,'temp.joblib',compress=3)

['temp.joblib']